In [44]:
import pandas as pd
import numpy as np

#-----------------------------------------
#Race setting and input variable selection 
#-----------------------------------------

NEMO_FILE = "Full Nemo.csv"

OUTCOME_VARIABLE = "pct_from_season_best"

BAD_RACE_IDS = [
    "1ab033a6-63dc-4611-b0fa-63a54e645cee"
]

JUNK_COMPETITIONS = [
    "Training",
    "CERTIFICATION 2022",
    "CERTIFICATION 2023",
    "CERTIFICATION 2025",
    "Certification 2026",
]

#Can be edited at any point
INPUT_VARIABLES = [
    #Overall pool volume
    "distance_10d",
    "distance_40d",
    #Number of pool sessions
    "sessions_10d",
    "sessions_40d",
    #Race-specific stroke volume
    "race_stroke_distance_10d",
    "race_stroke_distance_40d",
    #Intensity-specific volume
    "intensity_anaerobic_capacity_10d",
    "intensity_anaerobic_capacity_40d",
    "intensity_aerobic_power_10d",
    "intensity_aerobic_power_40d",
    "intensity_anaerobic_power_10d",
    "intensity_anaerobic_power_40d",
    "intensity_speed_10d",
    "intensity_speed_40d",
    #ACWR
    "taper_ratio",
    #CMJ features
    "cmj_jump_height_flight_time_mean_10d",
    "cmj_jump_height_flight_time_max_10d",
    "cmj_jump_height_flight_time_mean_40d",
    "cmj_jump_height_flight_time_max_40d",

    "cmj_peak_power_bm_mean_10d",
    "cmj_peak_power_bm_max_10d",
    "cmj_peak_power_bm_mean_40d",
    "cmj_peak_power_bm_max_40d",

    "cmj_concentric_mean_force_mean_10d",
    "cmj_concentric_mean_force_max_10d",
    "cmj_concentric_mean_force_mean_40d",
    "cmj_concentric_mean_force_max_40d",

    "cmj_eccentric_mean_force_mean_10d",
    "cmj_eccentric_mean_force_max_10d",
    "cmj_eccentric_mean_force_mean_40d",
    "cmj_eccentric_mean_force_max_40d",
    #OPTIONAL SHOULDER FEATURES
    #Uncomment these when running the smaller shoulder-specific model.

    #"shoulder_peak_vertical_force_mean_40d",
    #"shoulder_peak_vertical_force_max_40d",
    #"shoulder_force_at_100ms_mean_40d",
    #"shoulder_force_at_100ms_max_40d",
    #"shoulder_start_time_to_peak_force_mean_40d",
    #"shoulder_start_time_to_peak_force_max_40d",
    
    # OPTIONAL HRV AND READINESS FEATURES
    # Uncomment these for the smaller HRV-specific model.

    # Recent absolute HRV
    # "hrv_mean_7d",

    # Recent HRV relative to the athlete's longer baseline
    # "hrv_7d_vs_28d_pct",

    # Recent subjective sleep quality
    # "sleep_value_mean_7d",
]

#-------------------------------
#Commit (Training Load) Settings
#-------------------------------

COMMIT_FILE = "Commit 3.csv"

TRAINING_WINDOWS = [10, 40]

TRAINING_LOAD_VARIABLES = [
    "workout_total_distance",
    "intensity_anaerobic_capacity",
    "intensity_aerobic_power",
    "intensity_anaerobic_power",
    "intensity_speed",
]

STROKE_MAP = {
    "Freestyle": "stroke_free",
    "Backstroke": "stroke_back",
    "Breaststroke": "stroke_breast",
    "Butterfly": "stroke_fly",
    "Medley": "stroke_im",
}

INCLUDE_SESSION_COUNT = True
INCLUDE_STROKE_DISTANCE = True
INCLUDE_TAPER_RATIO = True

#--------------------
#FORCEDECKS SETTINGS
#--------------------

FORCEDECKS_FILE = "Forcedecks 3.csv"

#The modelling windows can be changed independently from the pool-training windows if required.
FORCEDECKS_WINDOWS = [10, 40]

#Using CMJ variables from past modelling for now. 
#Variables can be added to or removed from this list later.
CMJ_VARIABLES = [
    "Jump Height (Flight Time) [cm]",
    "Peak Power / BM [W/kg]",
    "Concentric Mean Force [N]",
    "Eccentric Mean Force [N]",
]

#Populate these after checking which variables have usable observations for the less frequently performed tests.

SHOULDER_ISO_VARIABLES = [
    "Peak Vertical Force [N]",      #Represents maximum force output
    "Force at 100ms [N]",          #Early force production
    "Start Time to Peak Force [s]",         #Time required to reach peak force
]
SINGLE_LEG_ISO_VARIABLES = []

#--------------------------
#HRV and Readiness Settings
#--------------------------

HRV_FILE = "HRV 3.csv"

HRV_VARIABLES = [
    "HRV",
    "Sleep Value",
]

HRV_WINDOWS = [3, 7, 28]

#Minimum number of valid daily observations required to calculate each rolling mean.

#These thresholds prevent a 28-day average from being calculated from
#only one or two readings while still allowing for occasional missingness.
HRV_MIN_OBSERVATIONS = {
    3: 2,
    7: 3,
    28: 14,
}

#Include the morning reading recorded on the race date.
INCLUDE_RACE_DAY_HRV = True

In [7]:
#Load in and clean races

cols_needed = [
    "RaceId",
    "Date",
    "AthleteName",
    "AthleteGenderName",
    "RaceEventName",
    "RaceEventDistance",
    "RaceEventSwimmingStyleName",
    "RaceEventIsRelay",
    "PoolLength",
    "PhaseName",
    "RaceTime",
    "CompetitionName",
]

df_nemo = pd.read_csv(
    NEMO_FILE,
    usecols=cols_needed,
    parse_dates=["Date"],
    dayfirst = True,
    low_memory=False
)

#Create unique identifier for every row
df_nemo["ModelRaceId"] = np.arange(len(df_nemo))

print(f"Original dataset: {df_nemo.shape}")
print(f"Missing RaceId: {df_nemo['RaceId'].isna().sum()}")
print(f"Missing CompetitionName: {df_nemo['CompetitionName'].isna().sum()}")

Original dataset: (3228, 13)
Missing RaceId: 391
Missing CompetitionName: 391


In [8]:
df_lc = df_nemo.copy()

#Keep individual races with valid race times
df_lc = df_lc[df_lc["RaceEventIsRelay"] == False]
df_lc = df_lc[df_lc["RaceTime"] > 0]

#Remove unwanted competitions and known error races
df_lc = df_lc[~df_lc["CompetitionName"].isin(JUNK_COMPETITIONS)]
df_lc = df_lc[~df_lc["RaceId"].isin(BAD_RACE_IDS)]

#Keep long-course races
df_lc = df_lc[df_lc["PoolLength"] == 50].copy()

print(f"Clean long-course dataset: {df_lc.shape}")
print(f"Rows removed: {len(df_nemo) - len(df_lc)}")

Clean long-course dataset: (2400, 13)
Rows removed: 828


In [9]:
def assign_season(date):
    if pd.isna(date):
        return pd.NA

    if date.month >= 9:
        start_year = date.year
    else:
        start_year = date.year - 1

    return f"{start_year}_{str(start_year + 1)[-2:]}"


df_lc["SeasonId"] = df_lc["Date"].apply(assign_season)

#Put races into the correct chronological order
df_lc = df_lc.sort_values(
    ["AthleteName", "RaceEventName", "Date", "ModelRaceId"]
).reset_index(drop=True)

print(df_lc["SeasonId"].value_counts(dropna=False).sort_index())
print(f"\nMissing dates: {df_lc['Date'].isna().sum()}")

SeasonId
2021_22    383
2022_23    497
2023_24    659
2024_25    482
2025_26    379
Name: count, dtype: int64

Missing dates: 0


In [10]:
def expanding_min_excluding_current(series):
    return series.shift(1).expanding().min()


#Best time achieved before each race in the current season
df_lc["season_best_so_far"] = (
    df_lc
    .groupby(
        ["AthleteName", "RaceEventName", "SeasonId"],
        sort=False
    )["RaceTime"]
    .transform(expanding_min_excluding_current))


#Full season best for each athlete and event
season_bests = (
    df_lc
    .groupby(
        ["AthleteName", "RaceEventName", "SeasonId"],
        as_index=False
    )["RaceTime"]
    .min()
    .rename(columns={"RaceTime": "full_season_best"}))


def previous_season(season_id):
    start_year = int(season_id.split("_")[0])
    previous_start = start_year - 1
    return f"{previous_start}_{str(start_year)[-2:]}"


season_bests["PreviousSeasonId"] = (
    season_bests["SeasonId"].apply(previous_season))


#Match each season to the immediately preceding season
prior_lookup = season_bests.merge(
    season_bests[
        [
            "AthleteName",
            "RaceEventName",
            "SeasonId",
            "full_season_best",
        ]
    ],
    left_on=[
        "AthleteName",
        "RaceEventName",
        "PreviousSeasonId",
    ],
    right_on=[
        "AthleteName",
        "RaceEventName",
        "SeasonId",
    ],
    how="left",
    suffixes=("", "_prior"),
)

prior_lookup = prior_lookup[
    [
        "AthleteName",
        "RaceEventName",
        "SeasonId",
        "full_season_best_prior",
    ]
].rename(
    columns={"full_season_best_prior": "prior_season_best"}
)


#Add the preceding season's best to each race
df_lc = df_lc.merge(
    prior_lookup,
    on=["AthleteName", "RaceEventName", "SeasonId"],
    how="left",
)

#Restore chronological order after merging
df_lc = df_lc.sort_values(
    ["AthleteName", "RaceEventName", "Date", "ModelRaceId"]
).reset_index(drop=True)


#Prefer the current season's best; otherwise use the prior season
df_lc["comparison_time"] = (
    df_lc["season_best_so_far"]
    .fillna(df_lc["prior_season_best"])
)

df_lc["comparison_source"] = np.select(
    [
        df_lc["season_best_so_far"].notna(),
        df_lc["prior_season_best"].notna(),
    ],
    [
        "current_season",
        "prior_season",
    ],
    default="none",
)


print(df_lc["comparison_source"].value_counts())
print(
    "\nRaces without a valid comparison:",
    df_lc["comparison_time"].isna().sum()
)

comparison_source
current_season    1997
prior_season       238
none               165
Name: count, dtype: int64

Races without a valid comparison: 165


In [13]:
#Create the outcome variable

df_lc["pct_from_season_best"] = (
    df_lc["RaceTime"] / df_lc["comparison_time"]
) * 100

df_lc["is_new_season_best"] = (
    df_lc["pct_from_season_best"] < 100
)

# Keep only races with a calculable outcome
df_model = df_lc[df_lc["comparison_time"].notna()].copy()

print(f"Final outcome dataset: {df_model.shape}")
print(df_model["pct_from_season_best"].describe())

print(
    "\nValues outside 90–110:",
    (
        (df_model["pct_from_season_best"] < 90)
        | (df_model["pct_from_season_best"] > 110)
    ).sum()
)

Final outcome dataset: (2235, 20)
count    2235.000000
mean      100.891779
std         1.797585
min        92.837135
25%        99.776328
50%       100.803653
75%       101.898779
max       112.086964
Name: pct_from_season_best, dtype: float64

Values outside 90–110: 2


In [19]:
###COMMIT

In [16]:
#Commit Inspection

df_commit = pd.read_csv(
    COMMIT_FILE,
    low_memory=False
)

print(f"Commit dataset: {df_commit.shape}")

df_commit[
    [
        "About",
        "Date",
        "workout_date",
        "workout_date_time",
        "workout_total_distance",
    ]
].head(10)

Commit dataset: (8669, 190)


,About,Date,workout_date,workout_date_time,workout_total_distance
0,Dean Fearn,11-07-2026,11-07-2026,2026-07-11 17:00:00+00:00,2500
1,Dean Fearn,11-07-2026,11-07-2026,2026-07-11 08:00:19+00:00,2500
2,Keanna MacInnes,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,5000
3,Katie Shanahan,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,4800
4,Lucy Grieve,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,4000
5,George Smith,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,5800
6,Holly McGill,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,5000
7,Jack McMillan,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,5800
8,Lucy Hope,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,5000
9,Thomas Dean,11-07-2026,11-07-2026,2026-07-11 07:00:00+00:00,4000


In [22]:
df_commit = df_commit.copy() #To defragment large dataframe

#Parse mixed timestamp formats and standardise
df_commit["TrainingTimestamp"] = pd.to_datetime(
    df_commit["workout_date_time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

#Clean athlete identifiers to match race data
df_commit["About"] = df_commit["About"].astype("string").str.strip()
df_model["AthleteName"] = df_model["AthleteName"].astype("string").str.strip()

#Ensure the selected training-load variables are numeric
for column in TRAINING_LOAD_VARIABLES:
    df_commit[column] = pd.to_numeric(
        df_commit[column],
        errors="coerce"
    ).fillna(0)

#Identify athlete coverage across the two datasets
commit_athletes = set(df_commit["About"].dropna())
race_athletes = set(df_model["AthleteName"].dropna())

#Report timestamp and athlete matching checks
print(
    f"Missing training timestamps: "
    f"{df_commit['TrainingTimestamp'].isna().sum()}"
)

print(
    f"Commit date range: "
    f"{df_commit['TrainingTimestamp'].min().date()} to "
    f"{df_commit['TrainingTimestamp'].max().date()}"
)

print(f"Missing workout IDs: {df_commit['workout_id'].isna().sum()}")
print(f"Athletes in outcome data: {len(race_athletes)}")
print(f"Athletes matched to Commit: {len(race_athletes & commit_athletes)}")
print(f"Athletes without a Commit match: {len(race_athletes - commit_athletes)}")

Missing training timestamps: 0
Commit date range: 2025-01-06 to 2026-07-11
Missing workout IDs: 0
Athletes in outcome data: 27
Athletes matched to Commit: 16
Athletes without a Commit match: 11


In [25]:
##Commit Features Function

def create_training_features(
    commit,
    athlete_name,
    race_date,
    race_stroke
):
    """
    Summarise an athlete's pool training before one race.

    Creates features over each period specified in
    TRAINING_WINDOWS (10d, 40d).
    """

    #Convert the race date to a UTC timestamp so it can be compared
    #with the UTC training timestamps without a timezone error
    
    race_timestamp = pd.Timestamp(race_date)

    if race_timestamp.tzinfo is None:
        race_timestamp = race_timestamp.tz_localize("UTC")
    else:
        race_timestamp = race_timestamp.tz_convert("UTC")

    #Select only the Commit rows belonging to this athlete.
    athlete_training = commit[
        commit["About"] == athlete_name
    ]

    features = {}

    #If the athlete has no Commit data, return missing values rather
    #than zeros. Zero would incorrectly imply that they did no training.
    athlete_has_data = not athlete_training.empty

    for days in TRAINING_WINDOWS:

        #Create the boundaries of the pre-race window.
        #The race date itself is excluded.
        window_start = race_timestamp - pd.Timedelta(days=days)

        #A window is considered usable only if the athlete has recorded
        #training data extending back to the beginning of that window.
        complete_window = (
            athlete_has_data
            and athlete_training["TrainingTimestamp"].min() <= window_start
        )

        if complete_window:

            #Select this athlete's sessions occurring within the window.
            window_data = athlete_training[
                (athlete_training["TrainingTimestamp"] >= window_start)
                & (athlete_training["TrainingTimestamp"] < race_timestamp)
            ]

            #Sum each selected training-load measure over the window.
            for column in TRAINING_LOAD_VARIABLES:

                #Give total distance "distance_10d" rather than "workout_total_distance_10d".
                feature_name = (
                    "distance"
                    if column == "workout_total_distance"
                    else column
                )

                features[f"{feature_name}_{days}d"] = (
                    window_data[column].sum()
                )

            if INCLUDE_SESSION_COUNT:
                #nunique() counts two same-day sessions separately when they have different workout IDs.
                features[f"sessions_{days}d"] = (
                    window_data["workout_id"].nunique())

            if INCLUDE_STROKE_DISTANCE:
                #Select the Commit stroke column corresponding to the stroke of the race being predicted.
                stroke_column = STROKE_MAP.get(race_stroke)

                if stroke_column in window_data.columns:
                    features[f"race_stroke_distance_{days}d"] = (
                        window_data[stroke_column].sum())
                else:
                    features[f"race_stroke_distance_{days}d"] = np.nan

        else:
            #An incomplete or unavailable window is missing—not zero.
            for column in TRAINING_LOAD_VARIABLES:
                feature_name = (
                    "distance"
                    if column == "workout_total_distance"
                    else column
                )
                features[f"{feature_name}_{days}d"] = np.nan

            if INCLUDE_SESSION_COUNT:
                features[f"sessions_{days}d"] = np.nan

            if INCLUDE_STROKE_DISTANCE:
                features[f"race_stroke_distance_{days}d"] = np.nan

    #Calculate taper ratio only when both selected windows are available.
    if INCLUDE_TAPER_RATIO:
        distance_10d = features.get("distance_10d")
        distance_40d = features.get("distance_40d")

        if (
            pd.notna(distance_10d)
            and pd.notna(distance_40d)
            and distance_40d > 0
        ):
            features["taper_ratio"] = (
                distance_10d / distance_40d
            ) * 100
        else:
            features["taper_ratio"] = np.nan

    return features

In [26]:
##Run the training load feature function once for every race in df_model.

#axis=1 means that the function is applied row by row rather than separately to each dataframe column.
training_features = df_model.apply(
    lambda row: pd.Series(
        create_training_features(
            commit=df_commit,
            athlete_name=row["AthleteName"],
            race_date=row["Date"],
            race_stroke=row["RaceEventSwimmingStyleName"],
        )
    ),
    axis=1,
)

##Add the newly calculated training features to the race dataset.

#pd.concat(..., axis=1) joins the two dataframes side by side while preserving the corresponding race-row index.
df_model = pd.concat(
    [df_model, training_features],
    axis=1
)

print(f"Dataset after adding Commit features: {df_model.shape}")

##Display the number of usable and missing observations for each newly created training feature.
training_feature_coverage = pd.DataFrame({
    "available": training_features.notna().sum(),
    "missing": training_features.isna().sum(),
})

print(training_feature_coverage)

Dataset after adding Commit features: (2235, 35)
                                  available  missing
distance_10d                            577     1658
intensity_anaerobic_capacity_10d        577     1658
intensity_aerobic_power_10d             577     1658
intensity_anaerobic_power_10d           577     1658
intensity_speed_10d                     577     1658
sessions_10d                            577     1658
race_stroke_distance_10d                577     1658
distance_40d                            547     1688
intensity_anaerobic_capacity_40d        547     1688
intensity_aerobic_power_40d             547     1688
intensity_anaerobic_power_40d           547     1688
intensity_speed_40d                     547     1688
sessions_40d                            547     1688
race_stroke_distance_40d                547     1688
taper_ratio                             547     1688


In [29]:
#Load forcedecks dataset. 
#(low_memory=False prevents pandas from assigning inconsistent data types when reading a dataset with many columns).
df_forcedecks = pd.read_csv(
    FORCEDECKS_FILE,
    low_memory=False
)

#Create a fresh defragmented copy before adding columns.
df_forcedecks = df_forcedecks.copy()

#Convert ForceDecks dates to UTC timestamps.

df_forcedecks["ForceDecksTimestamp"] = pd.to_datetime(
    df_forcedecks["Date"],
    format="mixed",      #allows more than one date format in the same column.
    dayfirst=True,
    utc=True,
    errors="coerce"          #converts unreadable dates to NaT to be identified 
)

df_forcedecks["About"] = (
    df_forcedecks["About"]
    .astype("string")   
    .str.strip()            #Remove leading and trailing spaces from athlete identifiers
)

#Count rows, unique test IDs and athletes for each test type (CMJ/SHLDISOI/SLISOT).
#To determine whether rows represent separate tests or trials.
test_summary = (
    df_forcedecks
    .groupby("Test Type")
    .agg(
        rows=("Test Type", "size"),
        unique_test_ids=("Id", "nunique"),
        athletes=("About", "nunique"),
        first_date=("ForceDecksTimestamp", "min"),
        last_date=("ForceDecksTimestamp", "max"),
    )
    .sort_values("rows", ascending=False)
)

print(f"ForceDecks dataset: {df_forcedecks.shape}")
print(
    f"Missing ForceDecks timestamps: "
    f"{df_forcedecks['ForceDecksTimestamp'].isna().sum()}"
)
print(test_summary)

ForceDecks dataset: (7326, 597)
Missing ForceDecks timestamps: 0
           rows  unique_test_ids  athletes                first_date  \
Test Type                                                              
CMJ        5359              989        15 2024-01-08 00:00:00+00:00   
SHLDISOI   1763              306        15 2024-10-29 00:00:00+00:00   
SJ          123               25        15 2025-04-28 00:00:00+00:00   
SLISOSQT     28                2         1 2024-11-06 00:00:00+00:00   
SLISOT       20                8         4 2026-05-27 00:00:00+00:00   
ISOT         12                2         2 2026-02-05 00:00:00+00:00   
SLSTICR      12                2         1 2024-11-06 00:00:00+00:00   
SLHJ          8                2         1 2025-06-19 00:00:00+00:00   

                          last_date  
Test Type                            
CMJ       2026-06-22 00:00:00+00:00  
SHLDISOI  2026-06-17 00:00:00+00:00  
SJ        2025-10-08 00:00:00+00:00  
SLISOSQT  2025-01-21 00:

In [33]:
# Search for columns whose names suggest that they contain direct
# force, RFD or impulse measurements from the shoulder test.
candidate_columns = [
    column for column in numeric_columns
    if any(
        term in column.lower()
        for term in ["force", "rfd", "impulse"]
    )
    # Exclude historical dashboard variables copied onto the test rows.
    and "hist" not in column.lower()
    and "cmj" not in column.lower()
]

nonzero_summary = []

for column in candidate_columns:

    # Convert the column to numeric again as a safeguard. Any text or
    # invalid entries are converted to NaN rather than causing an error.
    values = pd.to_numeric(
        shoulder_tests[column],
        errors="coerce"
    )

    # A value must be present and different from zero to be potentially
    # informative for modelling.
    nonzero_mask = values.notna() & (values != 0)

    nonzero_summary.append({
        "variable": column,

        # Number of individual rows containing a non-zero result
        "nonzero_rows": nonzero_mask.sum(),

        # Number of actual testing occasions containing a non-zero result
        "tests_with_nonzero_value": shoulder_tests.loc[
            nonzero_mask, "Id"
        ].nunique(),

        # A variable needs more than one distinct value to contain
        # information that a model could potentially use.
        "unique_nonzero_values": values[nonzero_mask].nunique(),
    })

shoulder_nonzero_availability = (
    pd.DataFrame(nonzero_summary)
    .sort_values(
        [
            "tests_with_nonzero_value",
            "unique_nonzero_values",
        ],
        ascending=False
    )
)

shoulder_nonzero_availability.head(30)

,variable,nonzero_rows,tests_with_nonzero_value,unique_nonzero_values
280,Absolute Impulse 200ms [Ns],1763,306,915
276,Absolute Impulse 150ms [Ns],1763,306,874
186,Peak Vertical Force [N],1763,306,863
70,Force at 200ms [N],1763,306,842
66,Force at 150ms [N],1763,306,823
272,Absolute Impulse 100ms [Ns],1763,306,797
268,Absolute Impulse 75ms [Ns],1763,306,770
62,Force at 100ms [N],1763,306,755
264,Absolute Impulse 50ms [Ns],1763,306,712
182,Start Time to Peak Force [s],1763,306,683


In [35]:
def aggregate_forcedecks_trials(data, test_type, variables):
    """
    Create one row per ForceDecks testing occasion while retaining
    both the mean and maximum result across its individual trials.
    """

    #Retain the requested test type and relevant columns.
    test_data = data[
        data["Test Type"] == test_type
    ][
        [
            "Id",
            "About",
            "ForceDecksTimestamp",
            *variables,
        ]
    ].copy()

    #Convert the selected measurements to numeric values.
    #Invalid text entries become NaN and are ignored in calculations.
    for column in variables:
        test_data[column] = pd.to_numeric(
            test_data[column],
            errors="coerce"
        )

    grouping_columns = [
        "Id",
        "About",
        "ForceDecksTimestamp",
    ]

    #For each test ID, calculate both the average trial result
    #and the highest trial result for every selected variable.
    test_measurements = (
        test_data
        .groupby(grouping_columns)[variables]
        .agg(["mean", "max"])
    )

    #Convert column names into simple names such as:
    test_measurements.columns = [
        f"{variable}__{summary}"
        for variable, summary in test_measurements.columns
    ]

    #Return the grouping columns to normal dataframe columns.
    test_level_data = test_measurements.reset_index()

    return test_level_data


#Produce one row per CMJ testing occasion.
fd_cmj_tests = aggregate_forcedecks_trials(
    data=df_forcedecks,
    test_type="CMJ",
    variables=CMJ_VARIABLES,
)

#Produce one row per shoulder-isometric testing occasion.
fd_shoulder_tests = aggregate_forcedecks_trials(
    data=df_forcedecks,
    test_type="SHLDISOI",
    variables=SHOULDER_ISO_VARIABLES,
)

print(f"CMJ testing occasions: {len(fd_cmj_tests)}")
print(f"Shoulder testing occasions: {len(fd_shoulder_tests)}")

CMJ testing occasions: 989
Shoulder testing occasions: 306


In [37]:
import re  #(regular expression module, cleans the variable names)

def simplify_variable_name(variable_name):
    """
    Convert a long ForceDecks heading into a readable Python column name.

    Example:
    "Peak Power / BM [W/kg]" becomes "peak_power_bm".
    """

    #Remove units enclosed in square brackets.
    simplified = re.sub(r"\[[^\]]*\]", "", variable_name)

    #Keep useful text inside normal brackets but remove the brackets.
    simplified = simplified.replace("(", "").replace(")", "")

    #Replace symbols and spaces with underscores.
    simplified = simplified.replace("/", " ")
    simplified = re.sub(r"[^A-Za-z0-9]+", "_", simplified)

    #Remove underscores from the beginning/end and use lowercase.
    return simplified.strip("_").lower()


def create_forcedecks_features(
    test_data,
    athlete_name,
    race_date,
    variables,
    test_prefix,
):
    """
    Summarise ForceDecks results recorded before one race.
    """

    #Convert the race date into a UTC timestamp so it can be compared with ForceDecks Timestamp.
    race_timestamp = pd.Timestamp(race_date)

    if race_timestamp.tzinfo is None:
        race_timestamp = race_timestamp.tz_localize("UTC")
    else:
        race_timestamp = race_timestamp.tz_convert("UTC")

    #Keep testing occasions belonging to the race athlete.
    athlete_tests = test_data[
        test_data["About"] == athlete_name
    ]

    features = {}

    #Identify when this particular ForceDecks test dataset begins.
    #This is different for CMJ and shoulder-iso testing.
    data_start = test_data["ForceDecksTimestamp"].min()

    for days in FORCEDECKS_WINDOWS:

        #Define beginning of the pre-race window.
        window_start = race_timestamp - pd.Timedelta(days=days)

        #A window is unavailable if the athlete has no data for this test
        #or if ForceDecks collection had not started by the window beginning.
        window_is_available = (
            not athlete_tests.empty
            and pd.notna(data_start)
            and data_start <= window_start
        )

        if not window_is_available:
            features[f"{test_prefix}_tests_{days}d"] = np.nan

            for variable in variables:
                short_name = simplify_variable_name(variable)

                features[
                    f"{test_prefix}_{short_name}_mean_{days}d"
                ] = np.nan

                features[
                    f"{test_prefix}_{short_name}_max_{days}d"
                ] = np.nan

            #Move directly to the next window without calculating false zeros.
            continue

        #Keep testing occasions inside the window while excluding
        #tests performed on the race date itself.
        window_tests = athlete_tests[
            (athlete_tests["ForceDecksTimestamp"] >= window_start)
            & (athlete_tests["ForceDecksTimestamp"] < race_timestamp)
        ]

        #Count actual testing occasions, not individual trials.
        features[f"{test_prefix}_tests_{days}d"] = len(window_tests)

        for variable in variables:

            short_name = simplify_variable_name(variable)

            #These columns were created when the trials within each
            #ForceDecks test ID were aggregated.
            test_mean_column = f"{variable}__mean"
            test_max_column = f"{variable}__max"

            #Calculate the average of the testing-occasion means.
            #If there are no tests, pandas returns NaN.
            features[
                f"{test_prefix}_{short_name}_mean_{days}d"
            ] = window_tests[test_mean_column].mean()

            #Calculate the highest trial maximum recorded across all
            #testing occasions in the pre-race window.
            features[
                f"{test_prefix}_{short_name}_max_{days}d"
            ] = window_tests[test_max_column].max()

    return features

In [38]:
##Add CMJ and Shoulder ISO to each race

def create_cmj_features_for_race(race_row):
    """
    Calculate CMJ features for one race.

    race_row contains all information from one row of df_model,
    including the athlete identifier and race date.
    """

    return pd.Series(
        create_forcedecks_features(
            #Use the dataset containing one row per CMJ test.
            test_data=fd_cmj_tests,

            #Match ForceDecks "About" to the race athlete.
            athlete_name=race_row["AthleteName"],

            #Use the race date as the end of each modelling window.
            race_date=race_row["Date"],

            #Calculate features only for the CMJ variables selected
            #in the settings section at the top of the notebook.
            variables=CMJ_VARIABLES,

            #Add "cmj" to the beginning of every generated feature.
            test_prefix="cmj",
        )
    )


def create_shoulder_features_for_race(race_row):
    """
    Calculate shoulder-isometric features for one race.
    """

    return pd.Series(
        create_forcedecks_features(
            #Use the dataset containing one row per shoulder test.
            test_data=fd_shoulder_tests,

            #Match the race athlete to the ForceDecks athlete.
            athlete_name=race_row["AthleteName"],

            #Include only testing completed before this race date.
            race_date=race_row["Date"],

            #Use the shoulder variables selected in the settings.
            variables=SHOULDER_ISO_VARIABLES,

            #Add "shoulder" to each generated feature name.
            test_prefix="shoulder",
        )
    )


#Apply the CMJ function separately to every race.

cmj_features = df_model.apply(
    create_cmj_features_for_race,
    axis=1,                   #tells pandas to work across rows instead of separately to columns
)


#Repeat the row-by-row calculation for shoulder-isometric testing.
shoulder_features = df_model.apply(
    create_shoulder_features_for_race,
    axis=1,
)


#Add the new feature columns beside the existing race and Commit columns.

df_model = pd.concat(
    [
        df_model,
        cmj_features,
        shoulder_features,
    ],
    axis=1,           #Joins side by side using the existing df indexes
)


print(f"Dataset after adding ForceDecks features: {df_model.shape}")

#Count races for which at least one actual testing occasion occurred
#within each pre-race window. NaN and zero are not counted.
print(
    "Races with at least one CMJ in previous 10 days:",
    (df_model["cmj_tests_10d"] > 0).sum()
)

print(
    "Races with at least one CMJ in previous 40 days:",
    (df_model["cmj_tests_40d"] > 0).sum()
)

print(
    "Races with at least one shoulder test in previous 10 days:",
    (df_model["shoulder_tests_10d"] > 0).sum()
)

print(
    "Races with at least one shoulder test in previous 40 days:",
    (df_model["shoulder_tests_40d"] > 0).sum()
)

Dataset after adding ForceDecks features: (2235, 67)
Races with at least one CMJ in previous 10 days: 584
Races with at least one CMJ in previous 40 days: 845
Races with at least one shoulder test in previous 10 days: 146
Races with at least one shoulder test in previous 40 days: 220


In [42]:
##LOAD AND INSPECT HRV DATASET

#Load the HRV and readiness export.
df_hrv = pd.read_csv(
    HRV_FILE,
    low_memory=False
)

#Create a fresh copy before adding columns.
df_hrv = df_hrv.copy()

#Convert daily record date into a consistent datetime format.

df_hrv["HRVDate"] = (
    pd.to_datetime(
        df_hrv["Date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )
    .dt.normalize()       #sets the time component to midnight because the records represent a day rather than a time event
)

#Remove accidental spaces around athlete names
df_hrv["About"] = (
    df_hrv["About"]
    .astype("string")
    .str.strip()
)

#Convert selected measures to numeric values.

#Any invalid text becomes NaN, while genuine numeric values are retained.
for column in HRV_VARIABLES:
    df_hrv[column] = pd.to_numeric(
        df_hrv[column],
        errors="coerce"
    )

print(f"HRV dataset: {df_hrv.shape}")
print(f"Missing dates: {df_hrv['HRVDate'].isna().sum()}")
print(f"Number of athletes: {df_hrv['About'].nunique()}")

print(
    "Date range:",
    df_hrv["HRVDate"].min().date(),
    "to",
    df_hrv["HRVDate"].max().date(),
)

#Check whether any athlete has more than one record on the same day.
print(
    "Duplicate athlete-date rows:",
    df_hrv.duplicated(
        subset=["About", "HRVDate"]
    ).sum()
)

#Show missingness for the initially selected variables.
print(
    "\nSelected-variable missingness:"
)

print(
    df_hrv[HRV_VARIABLES]
    .isna()
    .sum()
)

HRV dataset: (2013, 62)
Missing dates: 0
Number of athletes: 16
Date range: 2026-01-23 to 2026-07-13
Duplicate athlete-date rows: 0

Selected-variable missingness:
HRV            284
Sleep Value    311
dtype: int64


In [43]:
##Summarise the distributions of selected daily measures.

#Helps identify unexpected values (zeros, impossible questionnaire scores, wrong HRV scale)
print(
    df_hrv[HRV_VARIABLES]
    .describe()
)

#Identify the period during which HRV collection was operating.
hrv_start_date = df_hrv["HRVDate"].min()
hrv_end_date = df_hrv["HRVDate"].max()

#Identify athletes appearing in both the HRV and race datasets.
hrv_athletes = set(df_hrv["About"].dropna())
model_athletes = set(df_model["AthleteName"].dropna())

matched_hrv_athletes = hrv_athletes & model_athletes

#Count races occurring during the HRV collection period for athletes
#who appear in both datasets. This is the maximum potential coverage,
#actual coverage may be lower because individual daily values are missing.
potential_hrv_races = df_model[
    df_model["AthleteName"].isin(matched_hrv_athletes)
    & (df_model["Date"] >= hrv_start_date)
    & (df_model["Date"] <= hrv_end_date)
]

print(f"\nAthletes matched to HRV: {len(matched_hrv_athletes)}")
print(
    "Races during HRV collection for matched athletes:",
    len(potential_hrv_races)
)

               HRV  Sleep Value
count  1729.000000  1702.000000
mean      8.656275    63.061692
std       1.128496    19.726891
min       5.600000     0.000000
25%       7.800000    50.000000
50%       8.600000    64.000000
75%       9.400000    77.000000
max      11.800000   100.000000

Athletes matched to HRV: 16
Races during HRV collection for matched athletes: 253


In [46]:
##Define HRV rolling feature function

def create_hrv_features(
    hrv_data,
    athlete_name,
    race_date,
):
    """
    Calculate rolling HRV and sleep features before one race.
    """

    #Convert the race date to midnight without a timezone because HRVDate
    #also represents a date without a specific time.
    race_day = pd.Timestamp(race_date).normalize()

    #Keep only daily records belonging to this athlete.
    athlete_hrv = hrv_data[
        hrv_data["About"] == athlete_name
    ]

    #Identify the overall period covered by the HRV dataset.
    data_start = hrv_data["HRVDate"].min()
    data_end = hrv_data["HRVDate"].max()

    features = {}

    for days in HRV_WINDOWS:

        #If race-day HRV is included, a 3-day window consists of the
        #race day and the two preceding calendar days.
        window_end = (
            race_day
            if INCLUDE_RACE_DAY_HRV
            else race_day - pd.Timedelta(days=1)
        )

        window_start = (
            window_end - pd.Timedelta(days=days - 1)
        )

        #Check that the athlete exists in the HRV dataset and that the
        #dataset covers the entire requested calendar window.
        complete_window = (
            not athlete_hrv.empty
            and data_start <= window_start
            and data_end >= window_end
        )

        for variable in HRV_VARIABLES:

            #Convert names such as "Sleep Value" into "sleep_value".
            short_name = simplify_variable_name(variable)

            observation_name = (
                f"{short_name}_observations_{days}d"
            )
            mean_name = f"{short_name}_mean_{days}d"

            if not complete_window:
                #Missing coverage is recorded as NaN rather than zero.
                features[observation_name] = np.nan
                features[mean_name] = np.nan
                continue

            #Select this athlete's daily records inside the window.
            window_data = athlete_hrv[
                (athlete_hrv["HRVDate"] >= window_start)
                & (athlete_hrv["HRVDate"] <= window_end)
            ]

            #Count only non-missing observations for this variable.
            valid_observations = window_data[variable].dropna()

            observation_count = len(valid_observations)
            features[observation_name] = observation_count

            #Calculate the mean only when the window contains the
            #minimum number of valid observations specified above.
            if observation_count >= HRV_MIN_OBSERVATIONS[days]:
                features[mean_name] = valid_observations.mean()
            else:
                features[mean_name] = np.nan

    #Express recent HRV relative to the athlete's 28-day value.
    #Zero means the 7- and 28-day means are identical.
    hrv_7d = features.get("hrv_mean_7d")
    hrv_28d = features.get("hrv_mean_28d")

    if (
        pd.notna(hrv_7d)
        and pd.notna(hrv_28d)
        and hrv_28d != 0
    ):
        features["hrv_7d_vs_28d_pct"] = (
            (hrv_7d / hrv_28d) - 1
        ) * 100
    else:
        features["hrv_7d_vs_28d_pct"] = np.nan

    return features

In [47]:
##Apply HRV function to races

def create_hrv_features_for_race(race_row):
    """
    Extract athlete identifier and race date from one race row,
    calculate the corresponding pre-race HRV features.
    """

    return pd.Series(
        create_hrv_features(
            #Use the prepared daily HRV dataset.
            hrv_data=df_hrv,

            #Match HRV "About" to Nemo "AthleteName".
            athlete_name=race_row["AthleteName"],

            #Use this race's date to define the rolling windows.
            race_date=race_row["Date"],
        )
    )


#Apply the function separately to every row in the race dataset.

hrv_features = df_model.apply(
    create_hrv_features_for_race,
    axis=1,        #tells pandas that each item passed to function should be one complete race
)


#Join the generated HRV columns beside the existing race, Commit and ForceDecks columns.
df_model = pd.concat(
    [
        df_model,
        hrv_features,
    ],
    axis=1,
)


print(f"Dataset after adding HRV features: {df_model.shape}")

#Report how many races have usable values for the main HRV features.
print(
    "Races with a valid 3-day HRV mean:",
    df_model["hrv_mean_3d"].notna().sum()
)

print(
    "Races with a valid 7-day HRV mean:",
    df_model["hrv_mean_7d"].notna().sum()
)

print(
    "Races with a valid 28-day HRV mean:",
    df_model["hrv_mean_28d"].notna().sum()
)

print(
    "Races with a valid 7-day versus 28-day HRV comparison:",
    df_model["hrv_7d_vs_28d_pct"].notna().sum()
)

print(
    "Races with a valid 7-day sleep mean:",
    df_model["sleep_value_mean_7d"].notna().sum()
)

Dataset after adding HRV features: (2235, 80)
Races with a valid 3-day HRV mean: 161
Races with a valid 7-day HRV mean: 208
Races with a valid 28-day HRV mean: 195
Races with a valid 7-day versus 28-day HRV comparison: 173
Races with a valid 7-day sleep mean: 208
